In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
from data_collection import build_dataset
from data_selection import select_data, split_data, create_yolo_format, convert_labels_to_numeric_representation
import pandas as pd
from ultralytics import YOLO

Function which pulls and organizes the images. Starts with pulling all the pokemon api, which takes the longest. Followed by kaggle then huggingface. On my machine it takes approximtely 3-4 minutes

In [5]:
build_dataset()

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Data collection completed


In [6]:
df = select_data(sources='all', levels=[1])

Selected 2219 images across 12 Pokemon.


In [7]:
train, test, val = split_data(df)

print(train)

                                             image_path       label  \
100   /home/branchn/.Gtech/Pokemon-Visualization-Too...    Magikarp   
776   /home/branchn/.Gtech/Pokemon-Visualization-Too...  Butterfree   
1458  /home/branchn/.Gtech/Pokemon-Visualization-Too...     Scyther   
1699  /home/branchn/.Gtech/Pokemon-Visualization-Too...       Doduo   
2079  /home/branchn/.Gtech/Pokemon-Visualization-Too...  Butterfree   
...                                                 ...         ...   
1949  /home/branchn/.Gtech/Pokemon-Visualization-Too...      Meowth   
2126  /home/branchn/.Gtech/Pokemon-Visualization-Too...     Pikachu   
1264  /home/branchn/.Gtech/Pokemon-Visualization-Too...  Kangaskhan   
865   /home/branchn/.Gtech/Pokemon-Visualization-Too...  Butterfree   
1812  /home/branchn/.Gtech/Pokemon-Visualization-Too...     Snorlax   

           source  
100    PokemonAPI  
776    PokemonAPI  
1458       Kaggle  
1699  HuggingFace  
2079  HuggingFace  
...           ...  
1949  H

This following code puts the data into a Ultralytics format for yolo models. It essentially tags each image with a bounding box (the whole image is the bounding box) needed for yolo models, and stores it in a text file. The images are also copied to keep referencing in an ultralytics yaml easier. 


In [8]:
list_of_labels = None # us the follwing list to train for specific pokemon ["magikarp","pikachu", "meowth"]
print("The pokemon labels that are being used for training are: ")
print((df["label"].value_counts()))

numberic_labels_train = convert_labels_to_numeric_representation(train, list_of_labels)
numberic_labels_test = convert_labels_to_numeric_representation(test, list_of_labels)
numberic_labels_val = convert_labels_to_numeric_representation(val, list_of_labels)

create_yolo_format(train, 'train', numberic_labels_train)
create_yolo_format(test, 'test', numberic_labels_test)
create_yolo_format(val, 'val', numberic_labels_val)

The pokemon labels that are being used for training are: 
label
Pikachu       218
Scyther       213
Magikarp      202
Butterfree    197
Doduo         193
Snorlax       182
Lapras        176
Kangaskhan    173
Pidgey        170
Chansey       166
Eevee         166
Meowth        163
Name: count, dtype: int64
Creating YOLO format text files for training data...


Creating yolo data for train : 100%|██████████| 1775/1775 [00:00<00:00, 4113.24it/s]


Creating YOLO format text files for testing data...


Creating yolo data for test : 100%|██████████| 222/222 [00:00<00:00, 3182.67it/s]


Creating YOLO format text files for validation data...


Creating yolo data for val : 100%|██████████| 222/222 [00:00<00:00, 3366.28it/s]


In [ ]:
model = YOLO('yolov8m.pt')

import torch

print(torch.__version__)

#will need to set device to 0 if you have a gpu and 'cpu' if you want to train on cpu. Training on cpu will be very slow.
# ../Dataset/My_yolo_dataset/pokedata.yaml'
#'../Dataset/new_10k_data/Synthetic/data.yaml
# '../Dataset/custom_combined_data/data.yaml'
# model.train(data='../Dataset/new_10k_data/Synthetic/data.yaml', epochs=30, imgsz=640, device=[4,5], lr0=0.005, project="noahs_project_v3", name="noahs_yolo8_model_v3")
model.train(data='../Dataset/new_10k_data/Synthetic/data.yaml', epochs=30, imgsz=640, device=[4,5], lr0=0.005, batch=16,patience=10,workers=4, mosaic=1.0,hsv_s=0.7,hsv_v=0.4,scale=0.5,fliplr=0.5, project="noahs_project_final_models", name="noahs_yolo8_model_final")



2.10.0+cu126
New https://pypi.org/project/ultralytics/8.4.42 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.33 🚀 Python-3.11.5 torch-2.10.0+cu126 CUDA:4 (NVIDIA H100 PCIe, 81110MiB)
                                                      CUDA:5 (NVIDIA H100 PCIe, 81110MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../Dataset/new_10k_data/Synthetic/data.yaml, degrees=0.0, deterministic=True, device=4,5, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ra

In [67]:
#This will save the predictions Pokemon-Visualization-Tool/data_operations/runs/detect/predictions/test_predictions
#source="../Dataset/My_yolo_dataset/test/images/"

#model = YOLO("/home/branchn/.Gtech/Pokemon-Visualization-Tool/Noahs_implementation/data_operations/runs/detect/noahs_project_v2/noahs_yolo8_model_v23/weights/best.pt")
#model = YOLO("/home/branchn/.Gtech/Pokemon-Visualization-Tool/Noahs_implementation/data_operations/runs/detect/noahs_project_v3/noahs_yolo8_model_v3/weights/best.pt")
#model = YOLO("/home/branchn/.Gtech/Pokemon-Visualization-Tool/Noahs_implementation/data_operations/runs/detect/noahs_project_v3/noahs_yolo8_model_v33/weights/best.pt")
results = model.predict(source="../Dataset/test_video/Final_Test.mp4", project="predictions_v2/", name="test_custom_predictions_v2_conf_0.35", device=[4,5], conf=0.10, save=True , exist_ok=False)
#../Dataset/test_video/Final_Test.mp4
#results = model.predict(source="../Dataset/custom_test_images/", project="predictions_new_customdata2/", name="test_custom_predictions_new_custom_data2", conf=0.10, save=True , exist_ok=False)



WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/10132) /home/branchn/.Gtech/Pokemon-Visualization-Tool/Noahs_implementation/data_operations/../Dataset/test_video/Final_Test.mp4: 384x640 1 pikachu, 8.8ms
video 1/1 (frame 2/10132) /home/branchn/.Gtech/Pokemon-Visualization-Tool/Noahs_implementation/data_operations/../Dataset/test_video/Final_Test.mp4: 384x640 1 pikachu, 8.7ms
video 1/1 (frame 3/10132) /home/branchn/.Gtech/Pokemon-Visualization-Tool/Noahs_implementation/data_operations